# Financial Data Preprocessor (Beginner Version)

**Goal:** take a messy table of quarterly earnings and turn it into a clean table that is ready for analysis.

We will do it in 8 small steps. Every line of code has a comment (`# ...`) that explains what it does.
Run the cells **one by one** (Shift + Enter) and look at the output after each step.

| Step | What we do |
|---|---|
| 1 | Create the messy data |
| 2 | Look at the problems |
| 3 | Clean column names and tickers |
| 4 | Turn text like `"$90,000M"` into real numbers |
| 5 | Turn text like `"Q1 2023"` into real quarters |
| 6 | Remove duplicates |
| 7 | Fix outliers and fill missing values |
| 8 | Add new columns, check, and save |

## Step 0: Import the libraries

- **pandas** works with tables (called *DataFrames*).
- **numpy** does fast math on arrays of numbers.

In [ ]:
import pandas as pd   # pandas: tables (DataFrames). 'pd' is just a short nickname
import numpy as np    # numpy: math on numbers. 'np' is the usual nickname

## Step 1: Create the messy data

Real data is usually loaded with `pd.read_csv("file.csv")`. Here we type a small messy table by hand so you can see every problem clearly.

Problems hidden in this data:
- column names have spaces and symbols (`"Ticker "`, `"Revenue ($)"`)
- tickers have different cases and extra spaces (`" aapl"`)
- money is text, not numbers (`"$90,000M"`, `"$56.0B"`)
- some values are missing (`"N/A"`)
- quarters are written in different styles (`"Q1 2023"` vs `"2023-Q3"`)
- one row is duplicated (AAPL Q4 2023)
- one value is a huge mistake (`"$9,999,999M"`)

In [ ]:
# A dictionary: each key is a column name, each list is the values in that column
raw_data = {
    "Ticker ":       [" aapl", "AAPL", "AAPL", "AAPL", "AAPL ", "msft", "MSFT", "MSFT", "MSFT", "MSFT"],
    "Fiscal Period": ["Q1 2023", "Q2 2023", "2023-Q3", "Q4 2023", "Q4 2023",
                      "Q1 2023", "Q2 2023", "Q3 2023", "Q4 2023", "2024-Q1"],
    "Revenue ($)":   ["$90,000M", "$92,500M", "N/A", "$99,000M", "$99,000M",
                      "$50,000M", "$9,999,999M", "$54,000M", "$56.0B", "$58,000M"],
    "Net Income":    ["$20,000M", "$21,000M", "$22,500M", "N/A", "$24,000M",
                      "$15,000M", "$15,500M", "N/A", "$17,000M", "$17,500M"],
}

raw = pd.DataFrame(raw_data)   # turn the dictionary into a pandas table
raw                            # the last line of a cell is displayed automatically

## Step 2: Look at the problems

Always look at your data before changing it. `.info()` shows each column's type. Notice that **everything is `object` (text)**, even the money columns. We can't do math on text.

In [ ]:
raw.info()   # prints the column names, how many values, and the data type of each column

## Step 3: Clean the column names and tickers

- `.columns` is the list of column names.
- `.str.strip()` removes spaces at the start and end of text.
- `.str.upper()` makes text UPPERCASE.

In [ ]:
df = raw.copy()   # make a copy so the original 'raw' table stays untouched

# Give every column a simple, clean name
df.columns = ["ticker", "period", "revenue", "net_income"]

# Clean the ticker: remove extra spaces, then make it uppercase
df["ticker"] = df["ticker"].str.strip().str.upper()

df["ticker"].unique()   # show the different tickers that are left -> should be just AAPL and MSFT

## Step 4: Turn money text into numbers

We write a small function. A **function** is a reusable recipe: you give it an input, it gives back an output.

`money_to_number("$90,000M")` should return `90000.0` (in **millions of dollars**).

Rules:
- `"N/A"` becomes `NaN` (NaN = "Not a Number" = a missing value)
- `"B"` means billions, so we multiply by 1000 to get millions
- `"M"` means millions, so we just remove the letter
- `"(45)"` with brackets means a negative number

In [ ]:
def money_to_number(text):
    text = str(text).strip()                    # make sure it is text and remove outer spaces

    if text.upper() in ["N/A", "NA", "", "--"]: # these words all mean "missing"
        return np.nan                           # np.nan is how Python writes "missing number"

    is_negative = text.startswith("(")          # accountants write negatives like (45)

    for symbol in ["$", ",", "(", ")", " "]:    # remove every symbol we don't need
        text = text.replace(symbol, "")

    multiplier = 1                              # default: the number is already in millions
    if text.endswith("B"):                      # B = billions
        multiplier = 1000                       # 1 billion = 1000 million
        text = text[:-1]                        # remove the last letter "B"
    elif text.endswith("M"):                    # M = millions
        text = text[:-1]                        # remove the last letter "M"

    number = float(text) * multiplier           # convert the text to a real number
    return -number if is_negative else number  # make it negative if it had brackets


# Test the function on a few examples before using it on the whole table
print(money_to_number("$90,000M"))   # 90000.0
print(money_to_number("$56.0B"))     # 56000.0
print(money_to_number("(45.2)"))     # -45.2
print(money_to_number("N/A"))        # nan

In [ ]:
# .apply() runs our function on every value in the column
df["revenue"]    = df["revenue"].apply(money_to_number)
df["net_income"] = df["net_income"].apply(money_to_number)

df

## Step 5: Turn quarter text into real quarters

`"Q1 2023"` and `"2023-Q1"` mean the same thing. We convert both into one standard pandas **Period** (like `2023Q1`), which can be sorted correctly.

In [ ]:
def clean_period(text):
    text = text.strip().upper()          # remove spaces, make uppercase

    if text.startswith("Q"):             # style 1: "Q1 2023"
        quarter = text[1]                # the 2nd character is the quarter number -> "1"
        year = text[-4:]                 # the last 4 characters are the year -> "2023"
    else:                                # style 2: "2023-Q1"
        year = text[:4]                  # the first 4 characters are the year -> "2023"
        quarter = text[-1]               # the last character is the quarter number -> "1"

    return pd.Period(f"{year}Q{quarter}", freq="Q")   # build a standard quarter like 2023Q1


df["period"] = df["period"].apply(clean_period)   # run the function on every row
df

## Step 6: Remove duplicates

AAPL Q4 2023 appears twice. One copy has `net_income` missing, the other has it.
`groupby(["ticker", "period"]).first()` merges rows with the same ticker + quarter and keeps the **first non-missing value** in each column, so we lose nothing.

In [ ]:
print("Rows before:", len(df))

# Group rows that share the same ticker and period, then keep the first real value of each column
df = df.groupby(["ticker", "period"], as_index=False).first()

print("Rows after: ", len(df))
df

## Step 7: Fix outliers and fill missing values

**7a. Outliers.** MSFT revenue of 9,999,999 is obviously a mistake. Our rule: if a value is more than **5 times** the median of its company, treat it as missing.
(The *median* is the middle value, so it isn't thrown off by one huge number.)

We use `np.where(condition, value_if_true, value_if_false)` from NumPy.

In [ ]:
# For each row, get the median revenue of that row's company
median_revenue = df.groupby("ticker")["revenue"].transform("median")

# If revenue is more than 5x the median -> replace with NaN, otherwise keep it
df["revenue"] = np.where(df["revenue"] > 5 * median_revenue, np.nan, df["revenue"])

df[["ticker", "period", "revenue"]]

**7b. Count what is missing.** `.isna()` gives True/False for each cell, and `.sum()` counts the Trues.

In [ ]:
columns_to_fill = ["revenue", "net_income"]

print("Missing values before filling:")
print(df[columns_to_fill].isna().sum())

# Remember which cells were missing, so we can always tell filled values from real ones later
for col in columns_to_fill:
    df[col + "_was_missing"] = df[col].isna()

**7c. Fill the missing values.** Two methods, in order:

1. **Interpolation** - draw a straight line between the value before and the value after. If AAPL revenue is 92,500 then missing then 99,000, the gap becomes 95,750. We do this **per company** so AAPL numbers never leak into MSFT.
2. **Median fallback** - if a gap can't be interpolated (for example, no neighbours at all), use the company's median.

In [ ]:
for col in columns_to_fill:
    # 1) straight-line fill inside each company's own rows
    df[col] = df.groupby("ticker")[col].transform(lambda s: s.interpolate())

    # 2) anything still missing gets that company's median
    df[col] = df[col].fillna(df.groupby("ticker")[col].transform("median"))

print("Missing values after filling:")
print(df[columns_to_fill].isna().sum())

df

## Step 8: Add new columns, check the result, and save

- **net_margin** = net income / revenue (how much profit per dollar of sales)
- **revenue_growth** = percent change in revenue vs. the previous quarter, per company

In [ ]:
# NumPy: only divide when revenue is above 0, otherwise put NaN (avoids divide-by-zero)
df["net_margin"] = np.where(df["revenue"] > 0, df["net_income"] / df["revenue"], np.nan)

# pct_change() = (this quarter - last quarter) / last quarter, calculated separately for each company
df["revenue_growth"] = df.groupby("ticker")["revenue"].pct_change()

df

In [ ]:
# Safety checks: if any of these are False, Python stops with an error
assert df[["revenue", "net_income"]].isna().sum().sum() == 0, "There are still missing values!"
assert not df.duplicated(["ticker", "period"]).any(),         "There are still duplicate rows!"
assert (df["revenue"] > 0).all(),                             "Revenue should be positive!"
print("All checks passed. The data is clean!")

In [ ]:
df_to_save = df.copy()
df_to_save["period"] = df_to_save["period"].astype(str)   # store quarters as text like "2023Q1" in the CSV

df_to_save.to_csv("clean_quarterly_earnings.csv", index=False)   # index=False: don't save the row numbers
print("Saved to clean_quarterly_earnings.csv")

df_to_save

## Summary: what you learned

| Skill | Pandas / NumPy tool used |
|---|---|
| Build / load a table | `pd.DataFrame(...)`, `pd.read_csv(...)` |
| Rename columns | `df.columns = [...]` |
| Clean text | `.str.strip()`, `.str.upper()` |
| Convert data | `.apply(function)` |
| Remove duplicates | `.groupby(...).first()` |
| Find missing values | `.isna().sum()` |
| Detect and replace outliers | `np.where(...)`, `.median()` |
| Fill missing values | `.interpolate()`, `.fillna()` |
| Per-company calculations | `.groupby("ticker")` |
| New columns | `np.where(...)`, `.pct_change()` |
| Validate | `assert` |
| Save | `.to_csv(...)` |

**Next step:** replace the hand-typed `raw_data` with your own file using `raw = pd.read_csv("your_file.csv")` and adjust the column names in Step 3.